# Enchilada: minimal interface demo

This notebook walks through the `L1Data` / `Block` / `BlockResult` / `Wheel` contract using `EchoBlock`, a no-op block that prints its inputs and returns a sampling counter. No real waveform model, no MCMC — just enough to see the plumbing work.

## Setup

Run these commands in a terminal from the enchilada repository root:

```sh
uv sync --extra numeric-orbits --extra examples
uv run --no-sync jupyter lab examples/demo.ipynb
```

Select the kernel from that environment and run the cells from top to bottom. The `examples` extra supplies JupyterLab; `numeric-orbits` supplies the dependencies for the ephemeris section. If you also use an optional external backend such as WDM, install it after `uv sync`; launching with `--no-sync` preserves those installed packages.

## 1. Build the observed `L1Data`

One frozen object holds both the TDI arrays and the run settings everyone in this run agrees on.

In [ ]:
import numpy as np

from enchilada import BlockResult, L1Data, Wheel
from enchilada.testing import EchoBlock

rng = np.random.default_rng(0)
n_samples = 1024
channels = ("A", "E", "T")

observed = L1Data(
    channel_data={ch: rng.standard_normal(n_samples) for ch in channels},
    sample_rate_hz=0.1,
    channel_names=channels,  # num_time_samples read off the arrays
    tdi_generation="2.0",
    physical_observable="fractional_frequency",
)  # start_time_gps defaults to 0.0
observed

## 2. Long names and short shadows

Every derived quantity has two spellings. Use whichever reads better.

In [ ]:
print(
    f"observation_duration_s  = {observed.observation_duration_s} s     (Tobs = {observed.Tobs})"
)
print(
    f"sample_rate_hz          = {observed.sample_rate_hz} Hz   (fs   = {observed.fs})"
)
print(
    f"sample_interval_s       = {observed.sample_interval_s} s     (dt   = {observed.dt})"
)
print(
    f"num_time_samples        = {observed.num_time_samples}        (N    = {observed.N})"
)
print(
    f"frequency_resolution_hz = {observed.frequency_resolution_hz} Hz   (df   = {observed.df})"
)
print(
    f"nyquist_frequency_hz    = {observed.nyquist_frequency_hz} Hz   (fny  = {observed.fny})"
)
print(
    f"start_time_gps          = {observed.start_time_gps} s     (t0   = {observed.t0})"
)

In [ ]:
L1Data.aliases()

## 3. Typo catcher

Common misspellings point at the canonical spelling instead of failing silently.

In [ ]:
try:
    _ = observed.T_obs
except AttributeError as e:
    print(e)

The Wheel owns pristine observations, the full residual, covariance, a random stream, and a **ledger** of complete `BlockResult` snapshots. The caller must provide a complete `initial_block_result` when registering each block. Registration validates and stores it without drawing parameters or advancing the Wheel RNG. Each block's `sample` method receives data minus every other block, the current covariance, its accepted `BlockResult`, and the sampling random generator. It returns a complete `BlockResult` containing its signal and/or noise covariance together with `model_parameters` and `sampler_state`. EchoBlock only returns its counter in `sampler_state`; its absent signal contributes zero. The Wheel records that state and rebuilds the residual from the observations. Blocks retain fixed configuration; evolving state is read through `wheel.ledger[name]`.

In [ ]:
ucb = EchoBlock(name="ucb")
mbhb = EchoBlock(name="mbhb")

wheel = Wheel(observed, random_seed=0)
wheel.add(ucb, initial_block_result=BlockResult(sampler_state={"num_sample_calls": 0}))
wheel.add(mbhb, initial_block_result=BlockResult(sampler_state={"num_sample_calls": 0}))

wheel.run(num_cycles=3)

## 4. Read block state from the ledger

`wheel.ledger[name]` is a copied complete `BlockResult`, including `model_parameters`, `sampler_state`, and metadata. `wheel.contribution(block_name=name)` is a copied signal-array dictionary. `wheel.residual()` returns the full residual, and `wheel.residual(exclude_block_name=name)` returns the conditional input for that block. `wheel.noise_covariance` holds the separate covariance.

The same EchoBlock instance can be registered on another Wheel with an independent counter.

In [ ]:
print(
    f"ucb sampling calls: {wheel.ledger['ucb'].sampler_state['num_sample_calls']}, "
    f"mbhb sampling calls: {wheel.ledger['mbhb'].sampler_state['num_sample_calls']}"
)

full = wheel.residual()  # observed minus every block's signal contribution
print(f"full residual RMS on 'A': {np.sqrt(np.mean(full.channel_data['A'] ** 2)):.4f}")

## 5. Attaching the constellation ephemeris

Real data comes with the spacecraft positions it was produced with. That ephemeris rides on `L1Data.orbit_ephemeris` so every block builds its response from the *same* constellation — and `L1Data` checks at construction that the ephemeris actually spans the observation, catching epoch mismatches (GPS vs zero-based times) before any sampling starts.

`NumericOrbit` needs the `numeric-orbits` extra (`uv sync --extra numeric-orbits`). Here we tabulate a synthetic circular constellation; for real data use `NumericOrbit.from_hdf5` (LDC/Mojito files) or `NumericOrbit.from_lisaorbits`.

In [ ]:
from dataclasses import replace

from enchilada import NumericOrbit

# a synthetic 40-day ephemeris: three spacecraft on a 1 AU circle
orbital_radius_m = 1.495978707e11
sample_times_gps = np.linspace(0.0, 40 * 86400.0, 200)
spacecraft_positions_m = np.zeros(
    (3, sample_times_gps.size, 3)
)  # (spacecraft, time, xyz), ecliptic metres
for spacecraft_index in range(3):
    # spacecraft spaced ~0.0167 rad apart on the circle -> ~2.5e9 m arms
    orbital_phase = (
        2 * np.pi * sample_times_gps / (365.25 * 86400.0) + spacecraft_index * 0.0167
    )
    spacecraft_positions_m[spacecraft_index, :, 0] = orbital_radius_m * np.cos(
        orbital_phase
    )
    spacecraft_positions_m[spacecraft_index, :, 1] = orbital_radius_m * np.sin(
        orbital_phase
    )

orbit = NumericOrbit(
    sample_times_gps=sample_times_gps,
    spacecraft_positions_m=spacecraft_positions_m,
)
print(
    f"tabulated span: {orbit.time_range_gps}, "
    f"nominal arm length = {orbit.nominal_arm_length_m:.3e} m, "
    f"transfer frequency = {orbit.transfer_frequency_hz:.4f} Hz"
)

observed_with_orbit = replace(observed, orbit_ephemeris=orbit)  # spans the data: fine
print(
    "orbit attached; blocks read conditional_residual.orbit_ephemeris instead of building their own"
)

In [ ]:
# the consistency checks in action: a data span the ephemeris does not
# cover is rejected at construction, and out-of-span position queries refuse
# to extrapolate
try:
    replace(observed, orbit_ephemeris=orbit, start_time_gps=39.9 * 86400.0)
except ValueError as e:
    print(f"span check: {e}\n")

try:
    orbit.positions(times_gps=np.array([100 * 86400.0]))
except ValueError as e:
    print(f"extrapolation check: {e}")

## 6. Where next

- **A real (toy) fit** — [`examples/toy_fit.py`](toy_fit.py) runs two conjugate Gaussian-amplitude blocks plus a sampled inverse-gamma white-noise block. Its caller-side helpers draw genuine priors with a separate initialization RNG, and it reads `noise_covariance.noise_variance()`, and collects posterior samples from the ledger with `Wheel.run(..., on_cycle_complete=...)`.
- **Time, frequency, and WDM** — [`domain_translation.ipynb`](domain_translation.ipynb) translates data, covariance, and complete results, then lets Wheel prepare each block's chosen representation. Its setup installs the optional local WDM backend.
- **Your own block** — provide `name` and implement `sample(conditional_residual, noise_covariance, current_block_result, *, rng)` (`src/enchilada/block.py` documents the contract). Prepare a complete `initial_result`, then run `enchilada.testing.check_block(block_under_test=your_block, observed_data=toy_observed, initial_block_result=initial_result, num_cycles=2, initial_noise_covariance=None, random_seed=0)` before joining a shared campaign. This validates the supplied initial result and two sampling cycles; the optional initial covariance and random seed configure the test campaign.